# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/krishamerja1903/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [2]:
from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download
import pandas as pd
import numpy as np

HF_TOKEN = userdata.get("HF_TOKEN")

api = HfApi(token=HF_TOKEN)
user = api.whoami()

print("Hugging Face connected successfully!")
print("Logged in as:", user["name"])

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

march = pd.read_parquet(file_path)

available = march[
    march["gsc_data_available"].eq(True) &
    march["ga4_data_available"].eq(True)
].copy()

print("March rows:", len(march))
print("Available rows:", len(available))
print("Unique pages:", available["content_hash_id"].nunique())

Hugging Face connected successfully!
Logged in as: Krisha1903


fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

March rows: 9841378
Available rows: 364347
Unique pages: 63856


Existing data-loading code ✅
        ↓
Existing output:
March rows: 9841378
Available rows: 364347
Unique pages: 63856
        ↓
[TEXT] ← Baseline rule
        ↓
[CODE] ← Signal-check code
        ↓
2. Build the ranked queue

In [3]:
# Aggregate March data to page level
page = (
    available.groupby("content_hash_id", as_index=False)
    .agg(
        impressions=("gsc_impressions", "sum"),
        clicks=("gsc_clicks", "sum"),
        avg_position=("gsc_avg_position", "mean"),
        pageviews=("ga4_pageviews", "sum"),
        engaged_sessions=("ga4_engaged_sessions", "sum")
    )
)

# CTR
page["ctr"] = np.where(
    page["impressions"] > 0,
    page["clicks"] / page["impressions"],
    np.nan
)

# SIGNAL 1: impressions / visibility
page["visibility_bucket"] = pd.qcut(
    page["impressions"].rank(method="first"),
    4,
    labels=["LOW", "MEDIUM", "HIGH", "VERY_HIGH"]
)

signal1 = (
    page.groupby("visibility_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        median_clicks=("clicks", "median"),
        median_ctr=("ctr", "median")
    )
)

print("SIGNAL 1 — VISIBILITY")
print(signal1)
print("Verdict: CONFIRMED")

# SIGNAL 2: CTR vs position
page["position_bucket"] = pd.cut(
    page["avg_position"],
    bins=[0, 3, 10, 20, np.inf],
    labels=["1-3", "4-10", "11-20", "21+"]
)

signal2 = (
    page.groupby("position_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        median_impressions=("impressions", "median"),
        median_ctr=("ctr", "median")
    )
)

print("\nSIGNAL 2 — CTR VS POSITION")
print(signal2)
print("Verdict: CONFIRMED")

SIGNAL 1 — VISIBILITY
                       n  median_clicks  median_ctr
visibility_bucket                                  
LOW                15964            0.0    0.000000
MEDIUM             15964            0.0    0.000000
HIGH               15964            2.0    0.006849
VERY_HIGH          15964            9.0    0.003935
Verdict: CONFIRMED

SIGNAL 2 — CTR VS POSITION
                     n  median_impressions  median_ctr
position_bucket                                       
1-3               8091                70.0    0.006390
4-10             28002               122.0    0.005587
11-20            12179               127.0    0.003815
21+              14443               112.0    0.000000
Verdict: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [4]:
# Baseline action score:
# prioritise pages with meaningful visibility but below-typical CTR

eligible = page[
    (page["impressions"] > 0) &
    page["ctr"].notna()
].copy()

# Use observed medians as thresholds
imp_threshold = eligible["impressions"].median()
ctr_threshold = eligible["ctr"].median()

eligible["high_visibility"] = eligible["impressions"] >= imp_threshold
eligible["low_ctr"] = eligible["ctr"] <= ctr_threshold

# Simple transparent baseline score
eligible["score"] = (
    eligible["high_visibility"].astype(int) * 50 +
    eligible["low_ctr"].astype(int) * 50
)

eligible["reason_code"] = np.where(
    eligible["high_visibility"] & eligible["low_ctr"],
    "HIGH_VIS_LOW_CTR",
    "MONITOR"
)

eligible["action"] = np.where(
    eligible["reason_code"].eq("HIGH_VIS_LOW_CTR"),
    "REVIEW",
    "MONITOR"
)

# Tie-break higher scores using impressions
queue = eligible.sort_values(
    ["score", "impressions"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

# Public-safe output
output_cols = [
    "rank", "content_hash_id", "score", "reason_code",
    "action", "impressions", "clicks", "ctr", "avg_position"
]

# Write required CSV
import os
os.makedirs("work/outputs", exist_ok=True)

queue[output_cols].to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Impression threshold:", imp_threshold)
print("CTR threshold:", ctr_threshold)
print("Queue rows:", len(queue))
print("\nTOP 20:")
display(queue[output_cols].head(20))

Impression threshold: 106.0
CTR threshold: 0.0034129692832764505
Queue rows: 63856

TOP 20:


,rank,content_hash_id,score,reason_code,action,impressions,clicks,ctr,avg_position
0,1,content_e8a52cf3d5988c07,100,HIGH_VIS_LOW_CTR,REVIEW,244931,669,0.002731,15.008339
1,2,content_0e03de7680314cd5,100,HIGH_VIS_LOW_CTR,REVIEW,221310,720,0.003253,2.675217
2,3,content_36e53e9c707674fc,100,HIGH_VIS_LOW_CTR,REVIEW,194579,242,0.001244,32.766674
3,4,content_4ffe18112a5642e3,100,HIGH_VIS_LOW_CTR,REVIEW,186983,586,0.003134,2.331060
4,5,content_8d7d99f109e19aa2,100,HIGH_VIS_LOW_CTR,REVIEW,181942,286,0.001572,2.568135
5,6,content_44f34c0a90047651,100,HIGH_VIS_LOW_CTR,REVIEW,168160,15,0.000089,7.324954
6,7,content_3df3f32f3fd58dea,100,HIGH_VIS_LOW_CTR,REVIEW,137891,194,0.001407,23.251180
7,8,content_df47d1b976106de4,100,HIGH_VIS_LOW_CTR,REVIEW,124727,158,0.001267,24.123242
8,9,content_5e1c049f62e33b11,100,HIGH_VIS_LOW_CTR,REVIEW,117698,166,0.001410,18.113831
9,10,content_545bb6cc7081ded3,100,HIGH_VIS_LOW_CTR,REVIEW,117501,281,0.002391,2.630103


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Top-20 review

The top-ranked pages are review candidates because they combine high observed search visibility with CTR below the March median. These are decision-support recommendations, not evidence that changing metadata or content will cause traffic gains.

Confidence is higher when average position is relatively strong because low CTR is more actionable when a page is already ranking well. For weaker positions, ranking difficulty or search intent may explain the low CTR, so those recommendations have lower confidence.

A recommendation could be wrong if the page serves an unusual search intent, if aggregated position hides query-level differences, or if the observed March period is not representative.

In [5]:
top20 = queue.head(20).copy()

# Confidence note based on observed average position
top20["confidence_note"] = np.select(
    [
        top20["avg_position"] <= 10,
        top20["avg_position"] <= 20
    ],
    [
        "Higher confidence: visible in top 10 but CTR is low",
        "Medium confidence: moderate ranking with low CTR"
    ],
    default="Lower confidence: weak position may explain low CTR"
)

# What could invalidate the recommendation
top20["what_would_make_it_wrong"] = np.where(
    top20["avg_position"] <= 10,
    "Query intent or SERP features may explain the low CTR",
    "Low CTR may mainly reflect weak ranking rather than content or metadata"
)

review = top20[
    [
        "rank",
        "content_hash_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

display(review)

,rank,content_hash_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_e8a52cf3d5988c07,REVIEW,HIGH_VIS_LOW_CTR,Medium confidence: moderate ranking with low CTR,Low CTR may mainly reflect weak ranking rather...
1,2,content_0e03de7680314cd5,REVIEW,HIGH_VIS_LOW_CTR,Higher confidence: visible in top 10 but CTR i...,Query intent or SERP features may explain the ...
2,3,content_36e53e9c707674fc,REVIEW,HIGH_VIS_LOW_CTR,Lower confidence: weak position may explain lo...,Low CTR may mainly reflect weak ranking rather...
3,4,content_4ffe18112a5642e3,REVIEW,HIGH_VIS_LOW_CTR,Higher confidence: visible in top 10 but CTR i...,Query intent or SERP features may explain the ...
4,5,content_8d7d99f109e19aa2,REVIEW,HIGH_VIS_LOW_CTR,Higher confidence: visible in top 10 but CTR i...,Query intent or SERP features may explain the ...
5,6,content_44f34c0a90047651,REVIEW,HIGH_VIS_LOW_CTR,Higher confidence: visible in top 10 but CTR i...,Query intent or SERP features may explain the ...
6,7,content_3df3f32f3fd58dea,REVIEW,HIGH_VIS_LOW_CTR,Lower confidence: weak position may explain lo...,Low CTR may mainly reflect weak ranking rather...
7,8,content_df47d1b976106de4,REVIEW,HIGH_VIS_LOW_CTR,Lower confidence: weak position may explain lo...,Low CTR may mainly reflect weak ranking rather...
8,9,content_5e1c049f62e33b11,REVIEW,HIGH_VIS_LOW_CTR,Medium confidence: moderate ranking with low CTR,Low CTR may mainly reflect weak ranking rather...
9,10,content_545bb6cc7081ded3,REVIEW,HIGH_VIS_LOW_CTR,Higher confidence: visible in top 10 but CTR i...,Query intent or SERP features may explain the ...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Weak picks and leakage check

The weakest recommendations are generally pages with poor average search positions. For these pages, low CTR may be explained by limited ranking visibility rather than a content or metadata problem. Therefore, these recommendations should be treated as lower-confidence review candidates.

No future-window information, target-derived labels, or FlyRank product flags were used to calculate the baseline score. The score uses only observed March impressions and CTR. Average position is used for interpreting confidence in the Top-20 review, not for creating the baseline score.

This baseline is intentionally simple and transparent. It provides a decision-support benchmark that the later ML model can be compared against.

In [6]:
# Identify weaker Top-20 recommendations
weak_picks = top20[top20["avg_position"] > 20][
    ["rank", "content_hash_id", "impressions", "ctr", "avg_position"]
].copy()

print("Weak picks among Top 20:", len(weak_picks))
display(weak_picks)

# Leakage audit
score_inputs = ["impressions", "ctr"]

forbidden_inputs = [
    "future_clicks",
    "future_ctr",
    "decline_label",
    "product_flag"
]

used_forbidden = [x for x in forbidden_inputs if x in score_inputs]

print("\nLEAKAGE CHECK")
print("Score inputs:", score_inputs)
print("Future/label/product-flag inputs used:", used_forbidden)
print("Leakage check:", "PASS" if len(used_forbidden) == 0 else "FAIL")

Weak picks among Top 20: 9


,rank,content_hash_id,impressions,ctr,avg_position
2,3,content_36e53e9c707674fc,194579,0.001244,32.766674
6,7,content_3df3f32f3fd58dea,137891,0.001407,23.251180
7,8,content_df47d1b976106de4,124727,0.001267,24.123242
11,12,content_82e35c4845e6c391,102903,0.000525,21.904910
12,13,content_661a7734f691bef5,102735,0.000672,23.920025
14,15,content_cae701a83cad5e36,98572,0.002455,23.730705
15,16,content_b51957d7f4abe47e,85219,0.000669,28.809608
18,19,content_fa84f5976d5fe3c1,78716,0.001690,36.535261
19,20,content_89c10d52fc81ac39,75285,0.003214,21.921898



LEAKAGE CHECK
Score inputs: ['impressions', 'ctr']
Future/label/product-flag inputs used: []
Leakage check: PASS


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.